# Кредитный скоринг: Logistic Regression

**Сценарий:** банк хочет предсказывать `target_bad = 1`, серьёзную просрочку после выдачи кредита.

Модель здесь вызывается всего в трёх строках:

```python
model.fit(X_train, y_train)
pd_validation = model.predict_proba(X_validation)[:, 1]
auc = roc_auc_score(y_validation, pd_validation)
```

Основная работа: понять данные **до** этих строк и проверить результат **после** них.

## Цель

За 60 минут нужно пройти:

```text
audit → feature policy → feature engineering → time split
→ Logistic Regression → model validation → threshold
```

**Итог работы:** какие признаки оставлены, какие удалены, какая модель получена и какой threshold выбран.

## 0. Повторение: 5 минут

- Logistic Regression возвращает оценку `PD = P(bad=1 | features)`.
- `predict_proba`, а не `predict`, нужен для risk score и выбора threshold.
- Validation должна имитировать будущее: здесь используем time split.
Ответьте до написания кода:

1. Какой момент считается prediction moment?
2. Почему validation должна находиться позже train по времени?
3. Кто определяет threshold: модель или бизнес?

## Техническая справка

### Как выполнять notebook

1. Выполняйте ячейки последовательно сверху вниз.
2. Для запуска текущей ячейки используйте `Shift + Enter`.
3. Обозначение `[ ]` означает, что ячейка ещё не выполнялась.
4. Обозначение `[*]` означает, что ячейка выполняется.
5. Обозначение `[n]` означает, что ячейка успешно выполнилась.
6. Если появилась ошибка `NameError`, сначала проверьте, была ли выполнена ячейка, в которой создаётся соответствующая переменная.
7. Перед сдачей перезапустите kernel и выполните все ячейки сверху вниз через `Restart Kernel and Run All Cells`.

### Основные библиотеки

| Библиотека | Назначение |
|---|---|
| `pandas` | загрузка, очистка и анализ табличных данных |
| `numpy` | числовые операции и работа с массивами |
| `matplotlib` | построение графиков |
| `scikit-learn` | preprocessing, обучение модели и расчёт метрик |

Основные компоненты `scikit-learn`, используемые в задании:

| Компонент | Назначение |
|---|---|
| `SimpleImputer` | заполнение пропущенных значений |
| `StandardScaler` | стандартизация числовых признаков |
| `OneHotEncoder` | преобразование категориальных признаков |
| `ColumnTransformer` | применение разных преобразований к разным столбцам |
| `Pipeline` | объединение preprocessing и модели |
| `LogisticRegression` | оценка вероятности класса `target_bad = 1` |
| `roc_auc_score` | оценка качества ранжирования |
| `log_loss` | оценка качества прогнозируемых вероятностей |

### Обозначения `X` и `y`

В задачах машинного обучения обычно используются следующие обозначения:

```python
X = data[feature_columns]  # Матрица признаков
y = data["target_bad"]     # Целевая переменная
```

Каждая строка `X` соответствует одному наблюдению. Каждый столбец соответствует одному признаку. Вектор `y` содержит правильный класс для каждого наблюдения.

### `fit`, `predict` и `predict_proba`

Обучение модели:

```python
model.fit(X_train, y_train)
```

`fit` оценивает параметры модели по обучающей выборке.

Получение итогового класса:

```python
predicted_class = model.predict(X_validation)
```

Для стандартной бинарной Logistic Regression `predict` возвращает:

```text
0, если модель относит наблюдение к good
1, если модель относит наблюдение к bad
```

При стандартных настройках граница между классами соответствует вероятности около `0.50`.

Получение вероятностей:

```python
predicted_probabilities = model.predict_proba(X_validation)
```

Результат содержит две колонки:

```text
[:, 0] = P(target_bad = 0)
[:, 1] = P(target_bad = 1)
```

Для кредитного скоринга используется вероятность класса `bad`:

```python
pd_validation = model.predict_proba(X_validation)[:, 1]
```

Пример:

```text
predict_proba: [0.92, 0.08]

P(good) = 92%
P(bad)  = 8%
```

`predict` сразу переводит вероятность в класс. `predict_proba` сохраняет числовую оценку риска. Поэтому для скоринга, ранжирования клиентов и выбора бизнес-threshold используется `predict_proba`.

### Минимальная последовательность обучения

```python
model.fit(X_train, y_train)

train_pd = model.predict_proba(X_train)[:, 1]
validation_pd = model.predict_proba(X_validation)[:, 1]

train_auc = roc_auc_score(y_train, train_pd)
validation_auc = roc_auc_score(y_validation, validation_pd)
```

Сравнение train AUC и validation AUC показывает, насколько модель переносится на новые данные. Состав признаков и корректность временной постановки необходимо анализировать отдельно.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.style.use("seaborn-v0_8-whitegrid")

DATA_DIR = Path("credit_scoring_classwork.csv")
if not DATA_DIR.exists():
    DATA_DIR = Path("ml07") / "data"


def build_scoring_pipeline(numeric_features, categorical_features, C=1.0):
    numeric_pipeline = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipeline = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    preprocessor = ColumnTransformer(
        [
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features),
        ]
    )
    return Pipeline(
        [
            ("preprocessor", preprocessor),
            (
                "model",
                LogisticRegression(
                    max_iter=1000,
                    C=C,
                    solver="liblinear",
                ),
            ),
        ]
    )


def auc_report(model, X_train, y_train, X_validation, y_validation):
    train_pd = model.predict_proba(X_train)[:, 1]
    validation_pd = model.predict_proba(X_validation)[:, 1]
    train_auc = roc_auc_score(y_train, train_pd)
    validation_auc = roc_auc_score(y_validation, validation_pd)
    return {
        "train_auc": train_auc,
        "validation_auc": validation_auc,
        "auc_gap": train_auc - validation_auc,
        "validation_log_loss": log_loss(y_validation, validation_pd),
    }, train_pd, validation_pd

In [3]:
scoring_data = pd.read_csv("credit_scoring_classwork.csv")
print(scoring_data.shape)
scoring_data.head()

(12024, 20)


,application_id,application_date,age,monthly_income_uzs,requested_amount_uzs,employment_months,debt_to_income,bureau_score,active_loans,late_payments_12m,credit_history_months,channel,region,employment_type,has_salary_card,target_bad,requested_term_months,device_id,collection_calls_90d,max_dpd_90d
0,APP-011642,2025-12-22,26,"2,940,676.0000",13930404,9.0000,0.3240,589.3893,2,0,22.0000,mobile,Other,public_sector,yes,0,24,DEV-000F6AF5,0,0
1,APP-000883,2024-02-16,26,"6,908,969.0000",37001446,9.0000,0.3800,654.9562,0,0,47.0000,mobile,Tashkent,self_employed,no,0,36,DEV-000F46BC,0,0
2,APP-007195,2024-05-10,30,"6,406,539.0000",32959737,13.0000,0.3100,659.9332,0,1,76.0000,branch,Other,salaried,yes,0,12,DEV-000F441D,0,14
3,APP-000829,2024-10-20,49,"3,668,311.0000",9718265,22.0000,0.4940,596.8583,2,1,246.0000,web,Tashkent,salaried,no,0,36,DEV-000F62AD,0,0
4,APP-009549,2025-10-01,29,"3,783,563.0000",11674906,10.0000,0.0630,700.0169,0,0,59.0000,mobile,Samarkand,public_sector,no,0,9,DEV-000F4268,1,0


In [5]:
scoring_data

,application_id,application_date,age,monthly_income_uzs,requested_amount_uzs,employment_months,debt_to_income,bureau_score,active_loans,late_payments_12m,credit_history_months,channel,region,employment_type,has_salary_card,target_bad,requested_term_months,device_id,collection_calls_90d,max_dpd_90d
0,APP-011642,2025-12-22,26,"2,940,676.0000",13930404,9.0000,0.3240,589.3893,2,0,22.0000,mobile,Other,public_sector,yes,0,24,DEV-000F6AF5,0,0
1,APP-000883,2024-02-16,26,"6,908,969.0000",37001446,9.0000,0.3800,654.9562,0,0,47.0000,mobile,Tashkent,self_employed,no,0,36,DEV-000F46BC,0,0
2,APP-007195,2024-05-10,30,"6,406,539.0000",32959737,13.0000,0.3100,659.9332,0,1,76.0000,branch,Other,salaried,yes,0,12,DEV-000F441D,0,14
3,APP-000829,2024-10-20,49,"3,668,311.0000",9718265,22.0000,0.4940,596.8583,2,1,246.0000,web,Tashkent,salaried,no,0,36,DEV-000F62AD,0,0
4,APP-009549,2025-10-01,29,"3,783,563.0000",11674906,10.0000,0.0630,700.0169,0,0,59.0000,mobile,Samarkand,public_sector,no,0,9,DEV-000F4268,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12019,APP-004764,2024-05-31,53,"7,431,647.0000",27254188,17.0000,0.3820,663.0489,1,0,226.0000,web,Fergana,self_employed,no,0,36,DEV-000F4BD2,0,0
12020,APP-005001,2024-05-05,34,"4,152,053.0000",12193020,12.0000,0.0830,NaN,0,0,39.0000,web,Samarkand,salaried,no,0,12,DEV-000F6AE5,0,0
12021,APP-011582,2025-11-12,40,"5,620,420.0000",17843853,21.0000,0.3900,660.4346,0,1,42.0000,mobile,Other,salaried,yes,0,9,DEV-000F4A44,0,0
12022,APP-001862,2024-06-03,26,"4,423,330.0000",24417155,3.0000,0.5600,565.3854,3,1,46.0000,mobile,Tashkent,salaried,yes,0,12,DEV-000F429D,0,11


## 1. Audit данных: 10 минут

Проверьте:

1. размер, типы и bad rate;
2. период данных;
3. пропуски;
4. дубликаты `application_id`;
5. cardinality категорий;
6. в какой момент появляется каждый признак.

Сформируйте таблицу результатов audit и список признаков, выбранных для первой модели. Для каждого исключённого признака запишите основание исключения.

In [7]:
scoring_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 12024 entries, 0 to 12023
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   application_id         12024 non-null  str    
 1   application_date       12024 non-null  str    
 2   age                    12024 non-null  int64  
 3   monthly_income_uzs     11623 non-null  float64
 4   requested_amount_uzs   12024 non-null  int64  
 5   employment_months      11741 non-null  float64
 6   debt_to_income         12024 non-null  float64
 7   bureau_score           11368 non-null  float64
 8   active_loans           12024 non-null  int64  
 9   late_payments_12m      12024 non-null  int64  
 10  credit_history_months  12024 non-null  float64
 11  channel                12024 non-null  str    
 12  region                 12024 non-null  str    
 13  employment_type        12024 non-null  str    
 14  has_salary_card        12024 non-null  str    
 15  target_bad   

In [10]:
scoring_data["target_bad"].mean() * 100

np.float64(9.747172322022621)

In [14]:
scoring_data["application_date"] = pd.to_datetime(scoring_data["application_date"], format="%Y-%m-%d")

print(f'Данные начинается с {scoring_data["application_date"].min()} и продолжаются до {scoring_data["application_date"].max()}')

Данные начинается с 2024-01-01 00:00:00 и продолжаются до 2025-12-31 00:00:00


In [15]:
scoring_data.isna().sum()

application_id             0
application_date           0
age                        0
monthly_income_uzs       401
requested_amount_uzs       0
employment_months        283
debt_to_income             0
bureau_score             656
active_loans               0
late_payments_12m          0
credit_history_months      0
channel                    0
region                     0
employment_type            0
has_salary_card            0
target_bad                 0
requested_term_months      0
device_id                  0
collection_calls_90d       0
max_dpd_90d                0
dtype: int64

In [18]:
scoring_data.duplicated().sum()

np.int64(0)

In [19]:
scoring_data.nunique()

application_id           12000
application_date           731
age                         45
monthly_income_uzs       11569
requested_amount_uzs     11801
employment_months           56
debt_to_income             864
bureau_score             11347
active_loans                 7
late_payments_12m            7
credit_history_months      301
channel                      3
region                       4
employment_type              3
has_salary_card              2
target_bad                   2
requested_term_months        6
device_id                12000
collection_calls_90d        14
max_dpd_90d                106
dtype: int64

## 2. Feature policy и feature engineering: 10 минут

### Сначала очистите

- оставьте одну запись на `application_id`;
- при дубле сохраните самую раннюю заявку;
- исключите ID и признаки, недоступные в момент решения.

### Затем создайте три features

```text
requested_to_income
    = requested_amount_uzs / monthly_income_uzs

payment_to_income_proxy
    = (requested_amount_uzs / requested_term_months)
      / monthly_income_uzs

thin_file
    = credit_history_months < 12 или bureau_score пропущен
```

Вопрос: почему feature engineering нужно делать одинаково для train, validation и production?

In [4]:
# Ваш код

## 3. Time split и baseline: 10 минут

Граница:

```text
train      < 2025-07-01
validation ≥ 2025-07-01
```

Выполните следующие действия:

1. Разделите данные по времени.
2. Самостоятельно сформируйте списки числовых и категориальных признаков.
3. Обучите первую спецификацию Logistic Regression без подбора гиперпараметров.
4. Рассчитайте train AUC, validation AUC и validation Log Loss.
5. Сохраните состав признаков и полученные метрики.

In [5]:
# Ваш код

## 4. Диагностика модели: 15 минут

1. Сравните качество первой спецификации на train и validation.
2. Оцените, является ли полученный уровень качества реалистичным.
3. Повторно проверьте cardinality использованных признаков.
4. Повторно проверьте момент появления каждого признака относительно `application_date`.
5. Пересмотрите набор признаков при наличии сомнений.
6. Обучите вторую спецификацию модели.
7. Сравните две спецификации и объясните причину изменения метрик.

In [6]:
# Ваш код

## 5. Threshold и решение бизнеса: 10 минут

Для честного baseline проверьте thresholds:

```text
0.05, 0.08, 0.10, 0.15, 0.20
```

Заявка одобряется, если `PD < threshold`.

Посчитайте:

- approval rate;
- bad rate среди одобренных;
- долю пойманных bad;
- долю отклонённых good.

Выберите threshold и объясните компромисс одной фразой.

In [7]:
# Ваш код

## 6. Итог

Итог запишите в шести строках:

```text
1. Исключённые признаки и основания исключения: ...
2. Созданные признаки: ...
3. Первая спецификация, train AUC / validation AUC: ...
4. Вторая спецификация, train AUC / validation AUC: ...
5. Причина изменения качества: ...
6. Выбранный threshold и обоснование: ...
```

### Контрольные вопросы

1. Все ли признаки были доступны на prediction moment?
2. Есть ли существенный разрыв между train и validation?
3. Можно ли воспроизвести feature engineering в production?
4. Какие дополнительные проверки требуются перед внедрением?

## Следующий этап

На end-to-end scoring лекции добавим label maturity, calibration, LGD/EAD, expected loss, deployment и monitoring.